In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [3]:
# 1. Load only 3,000 rows for lab-speed training
df = pd.read_csv("IMDB Dataset.csv", nrows=3000)

# Convert sentiment into numerical labels
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df.head())
print(df.shape)

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1
(3000, 2)


In [4]:
# 3. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df["review"].tolist(),
    df["sentiment"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

In [5]:
# 4. Tokenize review texts using tiktoken cl100k_base encoding
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

train_tokens = [encoding.encode(text) for text in X_train]
test_tokens = [encoding.encode(text) for text in X_test]

In [6]:
# 5. Pad or truncate sequences to a fixed length
MAX_LEN = 200

def pad_or_truncate(sequences, max_len):
    result = []

    for seq in sequences:
        seq = seq[:max_len]

        if len(seq) < max_len:
            seq = seq + [0] * (max_len - len(seq))

        result.append(seq)

    return result

X_train_padded = pad_or_truncate(train_tokens, MAX_LEN)
X_test_padded = pad_or_truncate(test_tokens, MAX_LEN)

In [7]:
# 6. Convert token sequences and labels into tensors
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [8]:
# 7. Build model: Embedding -> RNN -> Linear output layer
VOCAB_SIZE = encoding.n_vocab
EMBED_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 2

class RNNClassifier(nn.Module):
    def __init__(self):
        super(RNNClassifier, self).__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            EMBED_DIM,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            EMBED_DIM,
            HIDDEN_DIM,
            batch_first=True
        )

        self.fc = nn.Linear(HIDDEN_DIM, OUTPUT_DIM)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.rnn(x)
        output = self.fc(hidden[-1])
        return output

model = RNNClassifier()

In [11]:
# 8. Train the model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100
BATCH_SIZE = 32

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for i in range(0, len(X_train_tensor), BATCH_SIZE):

        X_batch = X_train_tensor[i:i + BATCH_SIZE]
        y_batch = y_train_tensor[i:i + BATCH_SIZE]

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS}, "
        f"Loss: {total_loss / (len(X_train_tensor) / BATCH_SIZE):.4f}"
    )

Epoch 1/100, Loss: 0.4083
Epoch 2/100, Loss: 0.3934
Epoch 3/100, Loss: 0.3807
Epoch 4/100, Loss: 0.3648
Epoch 5/100, Loss: 0.3564
Epoch 6/100, Loss: 0.3518
Epoch 7/100, Loss: 0.3416
Epoch 8/100, Loss: 0.3233
Epoch 9/100, Loss: 0.3117
Epoch 10/100, Loss: 0.3046
Epoch 11/100, Loss: 0.3081
Epoch 12/100, Loss: 0.3011
Epoch 13/100, Loss: 0.3080
Epoch 14/100, Loss: 0.3151
Epoch 15/100, Loss: 0.3060
Epoch 16/100, Loss: 0.2928
Epoch 17/100, Loss: 0.2939
Epoch 18/100, Loss: 0.2870
Epoch 19/100, Loss: 0.3079
Epoch 20/100, Loss: 0.3120
Epoch 21/100, Loss: 0.2913
Epoch 22/100, Loss: 0.2912
Epoch 23/100, Loss: 0.2807
Epoch 24/100, Loss: 0.2793
Epoch 25/100, Loss: 0.2886
Epoch 26/100, Loss: 0.2976
Epoch 27/100, Loss: 0.3321
Epoch 28/100, Loss: 0.3081
Epoch 29/100, Loss: 0.2902
Epoch 30/100, Loss: 0.3527
Epoch 31/100, Loss: 0.3285
Epoch 32/100, Loss: 0.3559
Epoch 33/100, Loss: 0.3383
Epoch 34/100, Loss: 0.3045
Epoch 35/100, Loss: 0.2905
Epoch 36/100, Loss: 0.2847
Epoch 37/100, Loss: 0.2822
Epoch 38/1